<a href="https://colab.research.google.com/github/Nityakothavari7/EmberMind/blob/main/02_qwen_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q rouge-score bert-score sacrebleu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.7 MB/s eta 0:00:00


In [ ]:
import transformers
import peft
import torch

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Torch:", torch.__version__)

Transformers: 5.10.2
PEFT: 0.19.1
Torch: 2.11.0+cu128


In [ ]:
from transformers import BitsAndBytesConfig

print("OK")

OK


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_json(
    "/content/drive/MyDrive/ember_train_final.json"
)

train_df, test_df = train_test_split(
    df,
    test_size=0.05,
    random_state=42
)

print(len(train_df))
print(len(test_df))

16144
850


In [ ]:
train_df.to_json(
    "/content/drive/MyDrive/ember_train_split.json",
    orient="records"
)

test_df.to_json(
    "/content/drive/MyDrive/ember_test_split.json",
    orient="records"
)

In [ ]:
import torch
import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import PeftModel

from rouge_score import rouge_scorer
from bert_score import score as bertscore

import sacrebleu

In [ ]:
eval_df = test_df.sample(
    n=100,
    random_state=42
).reset_index(drop=True)

print("Evaluation Samples:", len(eval_df))

Evaluation Samples: 100


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
model = PeftModel.from_pretrained(
    base_model,
    "/content/drive/MyDrive/ember_qwen_lora"
)

model.eval()

print("EMBER-Qwen Loaded")

EMBER-Qwen Loaded


In [ ]:
def generate_response(user_input):

    prompt = f"""### User:
{user_input}

### Therapist:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    if "### Therapist:" in response:
        response = response.split(
            "### Therapist:"
        )[-1]

    return response.strip()

In [ ]:
predictions = []
references = []

for _, row in tqdm(
    eval_df.iterrows(),
    total=len(eval_df)
):

    pred = generate_response(
        row["input"]
    )

    predictions.append(pred)

    references.append(
        row["output"]
    )

print("Generation Complete")

  0%|          | 0/100 [00:00<?, ?it/s][transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 100/100 [1:02:03<00:00, 37.24s/it]

Generation Complete


In [ ]:
scorer = rouge_scorer.RougeScorer(
    ['rougeL'],
    use_stemmer=True
)

rouge_scores = []

for pred, ref in zip(
    predictions,
    references
):

    score = scorer.score(
        ref,
        pred
    )["rougeL"].fmeasure

    rouge_scores.append(score)

ROUGE_L = np.mean(
    rouge_scores
)

print("ROUGE-L:", ROUGE_L)

ROUGE-L: 0.21700588085253184


In [ ]:
BLEU = sacrebleu.corpus_bleu(
    predictions,
    [references]
)

print("BLEU:", BLEU.score)

BLEU: 10.362548791841068


In [ ]:
P, R, F1 = bertscore(
    predictions,
    references,
    lang="en",
    verbose=True
)

BERT_F1 = F1.mean().item()

print(
    "BERTScore:",
    BERT_F1
)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 10.57 seconds, 9.46 sentences/sec
BERTScore: 0.878451943397522


In [ ]:
import math

losses = []

for text in references:

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():
        outputs = model(
            **inputs,
            labels=inputs["input_ids"]
        )

    loss = outputs.loss.item()

    if not np.isnan(loss):
        losses.append(loss)

avg_loss = np.mean(losses)

print("Average Loss:", avg_loss)

try:
    perplexity = math.exp(avg_loss)
    print("Perplexity:", perplexity)
except OverflowError:
    print("Loss too large for stable perplexity calculation")

Average Loss: 1.396906529412125
Perplexity: 4.04267470826723


In [ ]:
print("\n========== EMBER-QWEN RESULTS ==========\n")

print(f"ROUGE-L     : {ROUGE_L:.4f}")
print(f"BLEU        : {BLEU.score:.4f}")
print(f"BERTScore   : {BERT_F1:.4f}")
print(f"Perplexity  : {perplexity:.4f}")


========== EMBER-QWEN RESULTS ==========

ROUGE-L     : 0.2170
BLEU        : 10.3625
BERTScore   : 0.8785
Perplexity  : 4.0427


In [ ]:
results_df = pd.DataFrame({
    "prediction": predictions,
    "reference": references
})

results_df.to_csv(
    "/content/drive/MyDrive/ember_eval_predictions.csv",
    index=False
)

In [ ]:
def chat_with_ember(user_input):

    prompt = f"""### User:
{user_input}

### Therapist:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    response = response.split(
        "### Therapist:"
    )[-1]

    return response.strip()

In [ ]:
while True:

    user_input = input("\nYou: ")

    if user_input.lower() in ["exit", "quit", "bye"]:
        print("EMBER-X: Goodbye!")
        break

    prompt = f"""### User:
{user_input}

### Therapist:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    response = response.split(
        "### Therapist:"
    )[-1].strip()

    print("\nEMBER-X:", response)


You: Hey, i feel like dying cause my mom made fun of my trauma and i feel like why did I even born??

EMBER-X: I'm sorry that you're feeling this way, but it's important to remember that everyone has their own unique experiences and challenges. It's natural to question your existence when going through difficult times, especially when someone close to you makes negative comments about your trauma.

It's essential to acknowledge that your feelings are valid, and it's okay to seek support from loved ones or professionals who can help you navigate these emotions. You might consider speaking with a trusted friend or family member about what your mom said, and let them know how their words affected you.

Additionally, exploring self-care activities such as exercise, meditation, or creative outlets like writing or painting could help you process your emotions and provide some relief during this challenging time.

Remember, you're not alone in this, and there are people who care about you an